# 03 - LoRA training (Qwen2.5-1.5B-Instruct)

GPU required. Install packages, clone the repo, regenerate the MedQuAD splits, train a LoRA adapter.

In [ ]:
%pip install -q -U torchao transformers datasets accelerate peft trl pyyaml

In [ ]:
import os
import sys

REPO_URL = "https://github.com/satyazm/Finetuning_LLMs.git"
REPO_DIR = "/kaggle/working/Finetuning_LLMs"

if not os.path.exists(REPO_DIR):
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    !git clone -q {clone_url} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

## Regenerate the dataset

Same as `02_baseline.ipynb`: `data/` isn't committed, so pull MedQuAD from the HF Hub again with the same seed/split.

In [ ]:
from src.data.preprocess import run as preprocess_run

preprocess_run(output_dir="data")

## Train

All the LoRA/training logic lives in `src/training/train_lora.py` (hyperparameters come from `configs/training.yaml` and `configs/model.yaml`). The notebook just calls `train()`.

In [ ]:
from src.training.train_lora import train

trainer = train(output_dir="adapters/lora")

## Verify

In [ ]:
import os

print(trainer.state.log_history[-1])
print(sorted(os.listdir("adapters/lora")))